# Statim: synthetic data test on Colab

**Before you start:** menu *Runtime → Change runtime type* → pick a **GPU** (T4 is the cheapest; L4 or A100 are faster but use more compute units) → Save. The setup cell prints the GPU it got and sets the parallel slots to fit it.

Then run the cells **one after another** (▶ on each cell, wait until it finishes):

| Cell | What it does | Time |
|---|---|---|
| Setup | GPU check, Ollama, Qwen3-8B + Phi-4-mini, Statim code | ~5–10 min |
| 1. Throughput | items per hour at 1/2/4/8 parallel requests | ~15–40 min |
| 2. Pilot | 2 × 200 examples + samples | ~1–3 h |
| 3. Save | copies the result to Google Drive | 1 min |

Licences: Qwen/Qwen3-8B (Apache-2.0) writes, microsoft/Phi-4-mini-instruct (MIT) checks blind; seed texts are BillSum (CC0), GovReport (CC-BY-4.0), EUR-Lex-Sum (CC-BY-4.0). Colab has no evaluation texts, so the output is marked *unchecked* and is filtered on pop-os (`tools/synth/leakage.py --filter`) before any training use.


In [ ]:
# ===== SETUP (once, ~8-12 min): GPU check, Ollama, models, Statim code =====
import os, subprocess, time, urllib.request, json, re, sys

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit("FAILED: " + cmd)
    return r.stdout

# 1. a GPU runtime is required
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, text=True, capture_output=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Menu: Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", gpu)

# 2. Ollama (official installer needs zstd)
sh("apt-get -qq update && apt-get -qq install -y zstd > /dev/null")
if subprocess.run("which ollama", shell=True).returncode:
    sh("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1")

# 3. parallel slots by GPU memory: both models must fit together (T4 15 GB -> 4 slots,
#    L4 24 GB -> 8, A100 40/80 GB -> 16); Colab keeps NVIDIA libs in /usr/lib64-nvidia
mem_mib = int(re.search(r"(\d+) MiB", gpu).group(1))
SLOTS = 16 if mem_mib >= 38000 else 8 if mem_mib >= 22000 else 4
print("parallel slots:", SLOTS)
# OLLAMA_CONTEXT_LENGTH: newer Ollama gives every slot 32k context on big GPUs (A100: phi4-mini alone 42 GB)
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(SLOTS), OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_CONTEXT_LENGTH="4096",
           OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="60m",
           LD_LIBRARY_PATH="/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", ""))
subprocess.run("pkill -x ollama", shell=True)
time.sleep(1)
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama", urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode()); break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start; see /tmp/ollama.log")

# 4. the two approved models (Qwen/Qwen3-8B Apache-2.0, microsoft/Phi-4-mini-instruct MIT)
for model in ("qwen3:8b", "phi4-mini"):
    sh(f"ollama pull {model} > /tmp/pull-{model.replace(':', '_')}.log 2>&1")
    print("pulled", model)

# 5. both models must run fully on the GPU ("100% GPU" in `ollama ps`)
sh('ollama run qwen3:8b "Say ok" > /dev/null 2>&1; ollama run phi4-mini "Say ok" > /dev/null 2>&1')
ps = subprocess.run("ollama ps", shell=True, text=True, capture_output=True).stdout
print(ps)
loaded = [line for line in ps.splitlines()[1:] if line.strip()]
if len(loaded) < 2 or not all("100% GPU" in line for line in loaded):
    print(open("/tmp/ollama.log").read()[-3000:])
    raise SystemExit("Both models must show 100% GPU in `ollama ps` (see above).")

# 6. Statim code (branch data/synth-v2) and Python packages
os.chdir("/content")  # a rerun may sit inside the folder it is about to replace
sh("rm -rf /content/statim && git clone -q --depth 1 -b data/synth-v2 https://github.com/BEKO2210/statim /content/statim")
sh("pip -q install datasets huggingface_hub pyarrow")
os.chdir("/content/statim")
print(sh("git log --oneline -1"))
print("SETUP OK")

In [ ]:
# ===== 1. THROUGHPUT (~15-40 min): kept items per hour at increasing parallel requests =====
# live: one progress bar per capability and level; at the end a table
LEVELS = ",".join(str(x) for x in (1, 4, SLOTS) if x <= SLOTS) if SLOTS > 4 else "1,2,4"
!cd /content/statim && TMPDIR=/tmp python -u tools/synth/bench_throughput.py --items 24 --levels {LEVELS} --cache /tmp/statim-synth-corpora --defer-leakage-check

In [ ]:
# ===== 2. PILOT (~30 min-2 h): 200 kept items each for urgency and NLI (8 languages) =====
# live: a progress bar per capability (kept/target, verifier agreement, kept per hour, ETA)
CONCURRENCY = SLOTS  # one request per server slot (set in Setup)
!cd /content/statim && TMPDIR=/tmp python -u tools/synth/grounded.py --per-capability 200 --concurrency {CONCURRENCY} --cache /tmp/statim-synth-corpora --out-dir /content/synth-colab-pilot --defer-leakage-check

import json, pandas as pd
from IPython.display import display, Markdown
m = json.load(open("/content/synth-colab-pilot/manifest.json"))
rows = []
for task, st in m["stats"].items():
    metrics = st["metrics"]
    rows.append({"capability": task, "kept": sum(st["counts"].values()), "seed jobs": st["jobs"],
                 "verifier agreement": "%.0f%%" % (100 * (st["verify_acceptance_rate"] or 0)),
                 "label word in text": metrics.get("label_leak", 0), "wrong language": metrics.get("bad_text", 0),
                 "copied from seed": metrics.get("seed_copy", 0)})
display(pd.DataFrame(rows))
print("minutes:", round(m["elapsed_s"] / 60, 1), "| kept per hour:", round(sum(r["kept"] for r in rows) * 3600 / m["elapsed_s"]))
display(Markdown(open("/content/synth-colab-pilot/samples.md").read()[:12000]))

In [ ]:
# ===== 3. SAVE to Google Drive (folder statim-synth) =====
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/statim-synth && cp -r /content/synth-colab-pilot /content/drive/MyDrive/statim-synth/
!ls -la /content/drive/MyDrive/statim-synth/synth-colab-pilot